In [ ]:
import torch
import wandb
import random
import os
import numpy as np

In [ ]:
def set_seed(seed=42):
    """
    Set random seeds to make experiments as reproducible as possible.
    """

    # Set the seed for Python's built-in random module
    random.seed(seed)

    # Set the seed for Python hash randomization
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Set the seed for NumPy
    np.random.seed(seed)

    # Set the seed for PyTorch on the CPU
    torch.manual_seed(seed)

    # Set the seed for PyTorch operations on the current GPU
    torch.cuda.manual_seed(seed)

    # Set the seed for all CUDA devices in a multi-GPU system
    torch.cuda.manual_seed_all(seed)

    # Make CuDNN operations deterministic
    # This improves reproducibility but may reduce performance.
    torch.backends.cudnn.deterministic = True

    # Disable CuDNN benchmarking to avoid selecting different algorithms
    # between runs.
    torch.backends.cudnn.benchmark = False

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """
    Train the model for one epoch.

    Args:
        model: PyTorch model.
        dataloader: Training data loader.
        optimizer: Optimizer used to update model parameters.
        criterion: Loss function.
        device: Device used for computation (CPU or GPU).

    Returns:
        Average training loss for the epoch.
    """

    # Set the model to training mode
    model.train()

    total_loss = 0.0

    # Iterate over all training batches
    for batch in dataloader:

        # Move the batch to the selected device
        batch = batch.to(device)

        # Clear gradients from the previous iteration
        optimizer.zero_grad()

        # Forward pass
        output = model(batch)

        # Calculate the loss
        loss = criterion(output, batch.y)

        # Backward pass
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate the batch loss
        total_loss += loss.item()

    # Calculate the average loss over all batches
    train_loss = total_loss / len(dataloader)

    return train_loss


def evaluate(model, dataloader, criterion, device):
    """
    Evaluate the model on validation or test data.

    Args:
        model: PyTorch model.
        dataloader: Validation or test data loader.
        criterion: Loss function.
        device: Device used for computation.

    Returns:
        Average evaluation loss and accuracy.
    """

    # Set the model to evaluation mode
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    # Disable gradient calculation during evaluation
    with torch.no_grad():

        # Iterate over all evaluation batches
        for batch in dataloader:

            # Move the batch to the selected device
            batch = batch.to(device)

            # Forward pass
            output = model(batch)

            # Calculate the loss
            loss = criterion(output, batch.y)

            # Accumulate the loss
            total_loss += loss.item()

            # Get the predicted class
            predictions = output.argmax(dim=1)

            # Count correct predictions
            correct += (predictions == batch.y).sum().item()

            # Count total samples
            total += batch.y.size(0)

    # Calculate average loss
    val_loss = total_loss / len(dataloader)

    # Calculate accuracy
    val_acc = correct / total

    return val_loss, val_acc

In [ ]:
# Define the hyperparameter sweep configuration
sweep_config = {

    # Bayesian optimization selects promising hyperparameter combinations
    # based on the results of previous runs.
    'method': 'bayes',  # Options: 'random', 'grid', or 'bayes'

    # Define the metric that W&B should optimize
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'  # Use 'maximize' for metrics such as accuracy
    },

    # Define the hyperparameters to search
    'parameters': {

        # Learning rate is sampled on a logarithmic scale
        # between approximately 1e-4 and 1e-2.
        'learning_rate': {
            'distribution': 'log_uniform_prior',
            'min': -9.21,  # exp(-9.21) ≈ 1e-4
            'max': -4.60   # exp(-4.60) ≈ 1e-2
        },

        # Test different batch sizes
        'batch_size': {
            'values': [16, 32, 64]
        },

        # Compare different optimization algorithms
        'optimizer': {
            'values': ['adam', 'adamw', 'sgd']
        },

        # Randomly sample dropout values from 0.1 to 0.5
        'dropout': {
            'distribution': 'uniform',
            'min': 0.1,
            'max': 0.5
        }
    }
}


# Create the sweep in Weights & Biases
sweep_id = wandb.sweep(
    sweep_config,
    project='BrainAge-GNN'
)

print("Sweep ID:", sweep_id)

In [ ]:
def main_sweep_pipeline():

    # Start a new W&B run.
    # W&B creates a new run and assigns hyperparameters
    # according to the sweep configuration.
    with wandb.init() as run:

        # Access the hyperparameters selected by W&B.
        config = wandb.config

        # Set the random seed before creating the model.
        # This makes random operations in the experiment
        # more reproducible.
        set_seed(config.seed)

        # Build the model using the selected hyperparameters.
        model = BuildMyModel(
            dropout=config.dropout
        )

        # Create the optimizer using the selected optimizer type
        # and learning rate.
        optimizer = GetOptimizer(
            config.optimizer,
            lr=config.learning_rate
        )

        # Initialize the best validation loss.
        # Infinity ensures that the first validation loss
        # will be considered the best result initially.
        best_val_loss = float('inf')

        # Define the path for saving the best model checkpoint.
        # The W&B run ID is used to create a unique filename.
        checkpoint_path = f"best_model_{run.id}.pt"

        # Training loop.
        for epoch in range(10):

            # Train the model for one epoch.
            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion
            )

            # Evaluate the model on the validation set.
            val_loss, val_acc = evaluate(
                model,
                val_loader,
                criterion
            )

            # Save the model if the current validation loss
            # is better than the best validation loss so far.
            if val_loss < best_val_loss:

                # Update the best validation loss.
                best_val_loss = val_loss

                # Save the model parameters.
                torch.save(
                    model.state_dict(),
                    checkpoint_path
                )

            # Log metrics to Weights & Biases.
            wandb.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc
            })

        # Create a W&B artifact for the best model.
        artifact = wandb.Artifact(
            name=f"model-{run.id}",
            type="model"
        )

        # Add the saved checkpoint file to the artifact.
        artifact.add_file(checkpoint_path)

        # Upload the model artifact to W&B.
        wandb.log_artifact(artifact)

In [ ]:
# Run the sweep with 15 different hyperparameter configurations
wandb.agent(
    sweep_id=sweep_id,
    function=main_sweep_pipeline,
    count=15
)